# AMLGuard: Anti-Money Laundering Detection under Extreme Class Imbalance

**Author:** Caio Bernardinelli  
**Date:** June 2026  
**Course:** Técnico em Inteligência Artificial — IFNMG  
**Objective:** Build an ML classifier to detect money laundering transactions
in the IBM AML dataset under extreme class imbalance (< 1% positive class).

---

## 1. Dataset Presentation

### Business Context

In 2025, the European Union centralized anti-money laundering enforcement
under the **AMLA (Authority for Anti-Money Laundering)**, operational since
July 2025 and based in Frankfurt. As of January 2026, AMLA assumed mandates
previously held by the EBA, and from 2028 will directly supervise ~40
high-risk cross-border institutions.

The core technical challenge: **illicit transactions represent less than 1%
of total volume** — a classic extreme class imbalance problem. Current systems
generate too many false positives, overwhelming compliance analysts.

### Dataset

**Source:** IBM Transactions for Anti-Money Laundering (AML)  
**Link:** https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml  
**License:** Community Data License Agreement - Sharing - Version 1.0  
**Version used:** HI-Small (High Illicit ratio — smallest version for rapid iteration)

### Variable Description

| Column | Type | Description |
|--------|------|-------------|
| Timestamp | object | Date and time of the transaction |
| From Bank | int64 | ID of the sending bank |
| Account | object | Account number of the sender |
| To Bank | int64 | ID of the receiving bank |
| Account.1 | object | Account number of the receiver |
| Amount Received | float64 | Amount received by the destination account |
| Receiving Currency | object | Currency of the received amount |
| Amount Paid | float64 | Amount paid by the source account |
| Payment Currency | object | Currency of the paid amount |
| Payment Format | object | Payment method (e.g. Cheque, Wire, Reinvestment) |
| Is Laundering | int64 | **Target variable** — 1 = illicit, 0 = legitimate |

---

In [1]:
# ============================================================================
# SETUP & IMPORTS
# ============================================================================

import warnings
warnings.filterwarnings('ignore')
import gdown

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    auc
)

# Imbalanced learning
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Explainability
import shap

# Global random state for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ All libraries imported successfully")
print(f"✓ Random state fixed: {RANDOM_STATE}")

✓ All libraries imported successfully
✓ Random state fixed: 42


## 2. Data Loading

The `HI-Small_Trans.csv` file is ~476 MB and is hosted on Google Drive rather than committed to the repository (the `.gitignore` excludes `*.csv`). It is downloaded at runtime with `gdown`, so the notebook is fully reproducible from a fresh clone: anyone running it fetches the data automatically, without needing the file in version control.

After loading, we run a first inspection — shape, column names, data types, memory footprint, and the distribution of the target variable — to confirm the data arrived intact and to quantify the class imbalance before any analysis.

In [10]:
# ============================================================================
# 2. DATA LOADING
# ============================================================================

print("="*80)
print("2. DATA LOADING")
print("="*80)

file_id = '1359N_tsRuUtCFMWCV6BtHjDdF280rb8e'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'HI-Small_Trans.csv', quiet=False)


df = pd.read_csv('HI-Small_Trans.csv')
print(df.shape)
print(df.columns.tolist())


# Basic inspection
print(f"\n✓ Dataset loaded successfully!")
print(f"\n{'─'*40}")
print(f"Shape:        {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"{'─'*40}")

print(f"\n--- First 10 rows ---")
print(df.head(10))

print(f"\n--- Data types ---")
print(df.dtypes)

print(f"\n--- Target variable (Is Laundering) ---")
print(df['Is Laundering'].value_counts())
print(f"\nIllicit transactions: {df['Is Laundering'].mean()*100:.4f}%")

2. DATA LOADING


Downloading...
From (original): https://drive.google.com/uc?id=1359N_tsRuUtCFMWCV6BtHjDdF280rb8e
From (redirected): https://drive.google.com/uc?id=1359N_tsRuUtCFMWCV6BtHjDdF280rb8e&confirm=t&uuid=24be5428-0a08-47e7-9151-262002e67a60
To: /content/HI-Small_Trans.csv
100%|██████████| 476M/476M [00:05<00:00, 81.9MB/s]


(5078345, 11)
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

✓ Dataset loaded successfully!

────────────────────────────────────────
Shape:        5,078,345 rows × 11 columns
Memory usage: 1890.96 MB
────────────────────────────────────────

--- First 10 rows ---
          Timestamp  From Bank    Account  To Bank  Account.1  \
0  2022/09/01 00:20         10  8000EBD30       10  8000EBD30   
1  2022/09/01 00:20       3208  8000F4580        1  8000F5340   
2  2022/09/01 00:00       3209  8000F4670     3209  8000F4670   
3  2022/09/01 00:02         12  8000F5030       12  8000F5030   
4  2022/09/01 00:06         10  8000F5200       10  8000F5200   
5  2022/09/01 00:03          1  8000F5AD0        1  8000F5AD0   
6  2022/09/01 00:08          1  8000EBAC0        1  8000EBAC0   
7  2022/09/01 00:16          1  8000EC1E0        1  8000EC1E0   
8  2022/09/01 00:26    

**Key finding — the imbalance is even more extreme than expected.**

The dataset holds **5,078,345 transactions** across 11 columns. Only **5,177** are labelled as laundering — **0.1019%** of the total, roughly one illicit transaction in every 980. This is well below the "under 1%" rule of thumb and sets the methodological constraints for the entire pipeline: stratified train/test split, precision–recall-oriented metrics (PR-AUC) instead of accuracy, and explicit handling of imbalance (class weighting / resampling) during modelling.

## 3. Descriptive Analysis

Before modelling, we characterise the data and test three hypotheses about *where* laundering concentrates. Given the extreme imbalance (base rate 0.1019%), the guiding principle throughout this section is to compare **illicit rate**, not raw counts: a frequent category will naturally contain more illicit cases in absolute terms simply because it contains more of everything. What signals risk is a group whose *proportion* of laundering sits well above the global base rate.

Each hypothesis below follows the same pattern — derive a flag, group by it, and read the mean of the target (which, for a 0/1 column, is the illicit rate of the group).

In [19]:
# ============================================================================
# 3. DESCRIPTIVE ANALYSIS
# ============================================================================

print("="*80)
print("3. DESCRIPTIVE ANALYSIS")
print("="*80)

# --- 3.1 Numeric overview ---
print("\n--- Amount Paid / Received: statistics ---")
print(df[['Amount Paid', 'Amount Received']].describe())

# --- 3.2 Categorical cardinality ---
print("\n--- Cardinality of categorical variables ---")
for col in ['Receiving Currency', 'Payment Currency', 'Payment Format', 'From Bank', 'To Bank']:
    print(f"{col}: {df[col].nunique()} unique values")

print("\n--- Payment Format distribution ---")
print(df['Payment Format'].value_counts())

print("\n--- Top 10 currencies (Payment) ---")
print(df['Payment Currency'].value_counts().head(10))

3. DESCRIPTIVE ANALYSIS

--- Amount Paid / Received: statistics ---
        Amount Paid  Amount Received
count  5.078345e+06     5.078345e+06
mean   4.509273e+06     5.988726e+06
std    8.697728e+08     1.037183e+09
min    1.000000e-06     1.000000e-06
25%    1.844800e+02     1.833700e+02
50%    1.414540e+03     1.411010e+03
75%    1.229784e+04     1.234627e+04
max    1.046302e+12     1.046302e+12

--- Cardinality of categorical variables ---
Receiving Currency: 15 unique values
Payment Currency: 15 unique values
Payment Format: 7 unique values
From Bank: 30470 unique values
To Bank: 15811 unique values

--- Payment Format distribution ---
Payment Format
Cheque          1864331
Credit Card     1323324
ACH              600797
Cash             490891
Reinvestment     481056
Wire             171855
Bitcoin          146091
Name: count, dtype: int64

--- Top 10 currencies (Payment) ---
Payment Currency
US Dollar      1895172
Euro           1168297
Swiss Franc     234860
Yuan            2137

### H1 — Cross-currency transactions

**Hypothesis:** moving money across currencies (paid ≠ received currency) could help shift wealth across borders and hinder tracing.

**Reading:** the hypothesis **cannot be tested on this dataset**. Cross-currency transactions are only 1.42% of all records (72,170 of 5.08M), and — decisively — **none** of them are labelled as laundering (0 illicit in 72,170). All 5,177 illicit cases are same-currency. This is a property of the synthetic generator, which never tags a cross-currency transaction as illicit, rather than an economic pattern. The flag therefore acts as a perfect *negative* predictor (cross-currency ⇒ certainly legitimate), which is a simulator artifact and not a generalizable signal. It will be **excluded** from modelling to avoid leakage, and noted as a dataset limitation in the conclusion.

In [26]:
df['is_cross_currency'] = (df['Payment Currency'] != df['Receiving Currency']).astype(int)
df.groupby('is_cross_currency')['Is Laundering'].agg(['mean', 'count', 'sum'])


,mean,count,sum
is_cross_currency,,,
0,0.001034,5006175,5177
1,0.000000,72170,0


In [27]:
print("--- Is_cross_currency distribuition ---")
print(df['is_cross_currency'].value_counts(normalize=True) * 100)


--- Is_cross_currency distribuition ---
is_cross_currency
0    98.578868
1     1.421132
Name: proportion, dtype: float64


### H2 — Payment format and laundering

**Hypothesis:** some payment methods offer easier laundering channels, so their illicit rate should exceed the global base rate (0.1019%).

**Reading:** only **ACH** stands out. Its illicit rate is **0.7462%** — about **7.3× the global base rate** — and it concentrates **86.6%** of all illicit cases. Crucially, the intuition that low-traceability channels (Bitcoin, Cash) would lead does **not** hold here: Bitcoin (0.0383%) and Cash (0.0220%) sit *below* the base rate, so a transaction in those formats is, if anything, *less* likely to be illicit than average. Their apparent prominence among illicit cases was a volume artifact (Reading B), not elevated risk (Reading A). Two formats — Reinvestment and Wire — contain **zero** illicit cases across hundreds of thousands of transactions, mirroring the generator-rule pattern seen in H1. ACH is therefore the only payment-format signal worth keeping as a feature, but its 7× concentration likely reflects how the synthetic generator routes laundering events rather than a real-world property of ACH, and is flagged as a dataset bias in the conclusion.

In [16]:
df.groupby('Payment Format')['Is Laundering'].agg(['mean', 'count', 'sum'])

,mean,count,sum
Payment Format,,,
ACH,0.007462,600797,4483
Bitcoin,0.000383,146091,56
Cash,0.000220,490891,108
Cheque,0.000174,1864331,324
Credit Card,0.000156,1323324,206
Reinvestment,0.000000,481056,0
Wire,0.000000,171855,0


In [28]:
fmt = df.groupby('Payment Format')['Is Laundering'].agg(['mean', 'count', 'sum'])

fmt = fmt.rename(columns={
    'mean':  'illicit_rate',     # illicit rate WITHIN the format (Reading A)
    'count': 'n_transactions',   # total volume of the format
    'sum':   'n_illicit'         # absolute illicit count
})

# Reading A — illicit rate within the format, as %
fmt['illicit_rate_%'] = (fmt['illicit_rate'] * 100).round(4)

# Legitimate rate within the format (complement of the illicit rate)
fmt['legit_rate_%'] = (100 - fmt['illicit_rate_%']).round(4)

# Reading B — how much each format represents of all illicit cases
fmt['%_of_all_illicit'] = (fmt['n_illicit'] / fmt['n_illicit'].sum() * 100).round(2)

# Context — how much each format represents of total volume
fmt['%_of_all_tx'] = (fmt['n_transactions'] / fmt['n_transactions'].sum() * 100).round(2)

fmt = fmt[['n_transactions', '%_of_all_tx', 'n_illicit',
           '%_of_all_illicit', 'illicit_rate_%', 'legit_rate_%']]
fmt = fmt.sort_values('illicit_rate_%', ascending=False)

print(f"Global base rate: {df['Is Laundering'].mean()*100:.4f}%\n")
print(fmt)

Global base rate: 0.1019%

                n_transactions  %_of_all_tx  n_illicit  %_of_all_illicit  \
Payment Format                                                             
ACH                     600797        11.83       4483             86.59   
Bitcoin                 146091         2.88         56              1.08   
Cash                    490891         9.67        108              2.09   
Cheque                 1864331        36.71        324              6.26   
Credit Card            1323324        26.06        206              3.98   
Reinvestment            481056         9.47          0              0.00   
Wire                    171855         3.38          0              0.00   

                illicit_rate_%  legit_rate_%  
Payment Format                                
ACH                     0.7462       99.2538  
Bitcoin                 0.0383       99.9617  
Cash                    0.0220       99.9780  
Cheque                  0.0174       99.9826  
Credit

### H3 — Same-bank vs. cross-bank transactions

**Hypothesis:** transactions between different banks/accounts could ease laundering, since an offender may push funds toward accounts where the money can be withdrawn with less direct traceability.

**Reading:** the direction holds, but the effect is **weak**. Cross-bank transactions show an illicit rate of **0.1157%** versus **0.0149%** for same-bank ones — so same-bank movement is ~7× *less* likely to be illicit, consistent with the idea that laundering favors moving funds across institutions. However, the cross-bank rate is only **1.14× the global base rate** (0.1019%), a modest lift compared with ACH's 7.3×. The "98% of illicit cases are cross-bank" figure is misleading on its own, since 86% of *all* transactions are already cross-bank; the real evidence is the rate increase, not the volume. `same_bank` is therefore a weak-but-valid feature, kept for modelling but not over-interpreted.

In [15]:
df['same_bank'] = (df['From Bank'] == df['To Bank']).astype(int)
df.groupby('same_bank')['Is Laundering'].agg(['mean', 'count', 'sum'])

,mean,count,sum
same_bank,,,
0,0.001157,4387013,5074
1,0.000149,691332,103


In [29]:
same_bank = df.groupby('same_bank')['Is Laundering'].agg(['mean', 'count', 'sum'])
same_bank = same_bank.rename(columns={'mean': 'illicit_rate', 'count': 'n_transactions', 'sum': 'n_illicit'})
same_bank['illicit_rate_%'] = (same_bank['illicit_rate'] * 100).round(4)
same_bank['%_of_all_tx'] = (same_bank['n_transactions'] / same_bank['n_transactions'].sum() * 100).round(2)
same_bank['%_of_all_illicit'] = (same_bank['n_illicit'] / same_bank['n_illicit'].sum() * 100).round(2)
same_bank = same_bank.rename(index={0: 'Different banks', 1: 'Same bank'})
print(f"Global base rate: {df['Is Laundering'].mean()*100:.4f}%\n")
print(same_bank[['n_transactions', '%_of_all_tx', 'n_illicit', '%_of_all_illicit', 'illicit_rate_%']])

Global base rate: 0.1019%

                 n_transactions  %_of_all_tx  n_illicit  %_of_all_illicit  \
same_bank                                                                   
Different banks         4387013        86.39       5074             98.01   
Same bank                691332        13.61        103              1.99   

                 illicit_rate_%  
same_bank                        
Different banks          0.1157  
Same bank                0.0149  


**Section summary**

Of the three hypotheses tested, only one produced a usable real signal, and a fourth lesson — methodological — runs through all of them: **always compare illicit *rate* within a group, never raw counts**, because frequent categories accumulate illicit cases simply by being frequent.

- **H1 — Cross-currency:** not testable on this dataset. Cross-currency transactions are 1.42% of the data and contain **zero** illicit cases, so the flag is a perfect *negative* predictor (cross-currency ⇒ certainly legitimate). This is a generator artifact, not an economic pattern — the feature is **excluded** to avoid leakage.
- **H2 — Payment format:** only **ACH** stands out, with an illicit rate of 0.7462% (**7.3× the base rate**) and 86.6% of all illicit cases. The low-traceability intuition (Bitcoin, Cash) did **not** hold — both sit *below* the base rate; their prominence was a volume artifact. ACH is kept as the strongest format signal, though its concentration likely reflects how the synthetic generator routes laundering events.
- **H3 — Same vs. cross-bank:** the hypothesized direction holds but the effect is **weak**. Cross-bank transactions reach 0.1157% illicit rate versus 0.0149% for same-bank — only **1.14× the base rate**. Kept as a weak-but-valid feature.

**Implication for modelling:** the global base rate of 0.1019% means accuracy is meaningless here; evaluation will rely on PR-AUC and recall at fixed precision. The strongest available signal (ACH) and the recurring presence of generator artifacts (zero-illicit groups) are both flagged as dataset limitations in the conclusion. Two formats (Reinvestment, Wire) and the entire cross-currency group contain no illicit cases at all — a reminder that on synthetic data, every strong signal must be questioned as *real pattern vs. simulator rule*.